In [58]:
import os, json, time, numpy as np, pandas as pd, dash
import plotly.express as px, plotly.graph_objs as go, dash_bootstrap_components as dbc
from dash import Input, Output, State, dcc, html, dash_table, ALL, MATCH, ctx

DATA_FOLDER = 'Dane Dash/' #Folder do zczytania gotowych danych
frames = []  
for filename in os.listdir(DATA_FOLDER):
    if filename.endswith('.json'):
        filepath = os.path.join(DATA_FOLDER, filename)
        frames.append(pd.read_json(filepath))
df = pd.concat(frames, ignore_index=True)
timestamp = df['timestamp']
timestamp_pl = timestamp.dt.tz_convert('Europe/Warsaw')
df['timestamp_pl'] = timestamp_pl
df['day_of_year'] = timestamp_pl.dt.day_of_year
df['weekday'] = timestamp_pl.dt.day_name('Pl')
df['weekday_number'] = timestamp_pl.dt.weekday
df['date'] = timestamp_pl.dt.date
df['localized_time_pl'] = df['timestamp_pl'].dt.tz_localize(None)

In [70]:
sdf = df[df['status']=='sleep'].reset_index(drop=True)
sdf['duration'] /= 3600
sdf.head(2)

,timestamp,app,status,duration,category,osoba,device,timestamp_pl,day_of_year,weekday,weekday_number,date,localized_time_pl
0,2026-01-01 00:17:00+00:00,Sen,sleep,8.65,sen,Antek,Phone,2026-01-01 01:17:00+01:00,1,Czwartek,3,2026-01-01,2026-01-01 01:17:00
1,2026-01-01 23:02:00+00:00,Sen,sleep,10.20,sen,Antek,Phone,2026-01-02 00:02:00+01:00,2,Piątek,4,2026-01-02,2026-01-02 00:02:00


### Violin Plot

In [71]:
sdf.groupby(['osoba','day_of_year'])['duration'].sum().reset_index(drop=False)

,osoba,day_of_year,duration
0,Antek,1,8.650000
1,Antek,2,10.299722
2,Antek,3,10.916667
3,Antek,4,9.500000
4,Antek,5,8.699722
...,...,...,...
135,Kamil,361,7.833333
136,Kamil,362,7.766667
137,Kamil,363,8.666667
138,Kamil,364,10.033333


In [72]:
sdf.loc[sdf.timestamp_pl.dt.hour > 18,'day_of_year'] += 1
sdf.loc[sdf.day_of_year == 366,'day_of_year'] = 1
sdf2 = sdf.groupby(['osoba','day_of_year'])['duration'].sum().reset_index(drop=False)

In [92]:
fig = px.violin(sdf2,x='osoba',y='duration',title='Rozkład czasu snu')
fig.update_layout(yaxis_title='liczba godzin',template='plotly_white',
                  xaxis_title="")